# Lakehouse Browser

Use this notebook to browse Iceberg tables in the local `silver` and `gold` catalogs.

In [1]:
from spark.lakehouse_browser import connect_browser_spark, describe_table, list_namespaces, list_tables, preview_table

spark = connect_browser_spark()
spark

/home/airflow/.local/lib/python3.12/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


:: loading settings :: url = jar:file:/home/airflow/.local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/airflow/.ivy2
The jars for the packages stored in: /home/airflow/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ae4222f6-138c-477d-a045-1ba2155eb499;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.5.2/iceberg-spark-runtime-3.5_2.12-1.5.2.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2!iceberg-spark-runtime-3.5_2.12.jar (1489ms)
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3

In [2]:
list_namespaces(spark, "silver")

26/03/22 17:52:00 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


,namespace
0,crypto


In [3]:
list_tables(spark, "silver", "crypto")

,namespace,tableName,isTemporary
0,crypto,coins,False
1,crypto,price_snapshots,False
2,crypto,total_market_cap_history,False
3,crypto,asset_market_cap_history,False


In [10]:
preview_table(spark, "silver", "crypto", "coins", limit=200)

,id,symbol,name,rank,supply,max_supply
0,binance-coin,BNB,BNB,5,1.363576e+08,1.363576e+08
1,bitcoin,BTC,Bitcoin,1,2.000304e+07,2.100000e+07
2,bitcoin-cash,BCH,Bitcoin Cash,13,2.001050e+07,2.100000e+07
3,cardano,ADA,Cardano,12,3.609644e+10,4.500000e+10
4,chainlink,LINK,Chainlink,19,7.081000e+08,1.000000e+09
5,dogecoin,DOGE,Dogecoin,10,1.535076e+11,NaN
6,ethereum,ETH,Ethereum,2,1.206918e+08,NaN
7,hyperliquid,HYPE,Hyperliquid,11,2.567791e+08,9.616715e+08
8,lido-finance-wsteth,WSTETH,Lido wstETH,14,3.490818e+06,NaN
9,monero,XMR,Monero,18,1.844674e+07,NaN


In [14]:
preview_table(spark, "silver", "crypto", "price_snapshots", limit=10000, sort="snapshot_date")

TypeError: preview_table() got an unexpected keyword argument 'sort'

In [19]:
spark.sql("""
SELECT *
FROM silver.crypto.price_snapshots
WHERE coin_id = 'bitcoin'
ORDER BY snapshot_date DESC
LIMIT 10000
""").toPandas()

,coin_id,snapshot_date,price_usd,market_cap_usd,volume_usd_24hr,change_percent_24hr,vwap_24hr
0,bitcoin,2026-03-21,69023.40000,1.380678e+12,2.167157e+10,-2.070715,70448.873064
1,bitcoin,2026-03-20,69023.40000,1.380678e+12,2.167157e+10,-2.070715,70448.873064
2,bitcoin,2026-03-19,69600.10000,1.392214e+12,3.189609e+10,0.198669,70269.131348
3,bitcoin,2026-03-18,71179.71043,1.423811e+12,3.761150e+10,-3.960453,72447.978671
4,bitcoin,2026-03-17,73918.29000,1.478591e+12,3.894624e+10,-1.099848,74397.843722
...,...,...,...,...,...,...,...
62,bitcoin,2026-01-18,95132.71000,1.901066e+12,NaN,NaN,NaN
63,bitcoin,2026-01-17,95552.91000,1.908266e+12,NaN,NaN,NaN
64,bitcoin,2026-01-16,95581.25000,1.911223e+12,NaN,NaN,NaN
65,bitcoin,2026-01-15,96944.66000,1.935422e+12,NaN,NaN,NaN


In [17]:
spark.sql("""
SELECT coin_id, min(snapshot_date) AS first_date, max(snapshot_date) AS last_date, count(*) AS rows
FROM silver.crypto.price_snapshots
GROUP BY coin_id
ORDER BY coin_id
""").toPandas()

,coin_id,first_date,last_date,rows
0,binance-coin,2026-01-14,2026-03-21,67
1,bitcoin,2026-01-14,2026-03-21,67
2,bitcoin-cash,2026-01-14,2026-03-21,67
3,cardano,2026-01-14,2026-03-21,67
4,chainlink,2026-01-14,2026-03-21,67
5,dogecoin,2026-01-14,2026-03-21,67
6,ethereum,2026-01-14,2026-03-21,67
7,hyperliquid,2026-01-14,2026-03-21,67
8,lido-finance-wsteth,2026-01-14,2026-03-21,67
9,monero,2026-01-14,2026-03-21,67


In [18]:
spark.sql("""
SELECT
  sum(CASE WHEN price_usd IS NULL THEN 1 ELSE 0 END) AS null_price,
  sum(CASE WHEN market_cap_usd IS NULL THEN 1 ELSE 0 END) AS null_market_cap
FROM silver.crypto.price_snapshots
""").toPandas()

,null_price,null_market_cap
0,0,0


In [15]:
spark.sql("SHOW TABLES IN silver.crypto").toPandas()

,namespace,tableName,isTemporary
0,crypto,coins,False
1,crypto,price_snapshots,False
2,crypto,total_market_cap_history,False
3,crypto,asset_market_cap_history,False


In [6]:
describe_table(spark, "silver", "crypto", "price_snapshots")

,col_name,data_type,comment
0,coin_id,string,FK → silver.crypto.coins.id
1,snapshot_date,date,Logical execution date (UTC)
2,price_usd,double,USD price at snapshot time
3,market_cap_usd,double,Total market cap in USD
4,volume_usd_24hr,double,Trading volume over last 24 h
5,change_percent_24hr,double,Price change % over last 24 h
6,vwap_24hr,double,Volume-weighted avg price (24 h)
7,# Partition Information,,
8,# col_name,data_type,comment
9,snapshot_date,date,Logical execution date (UTC)


When Gold tables exist, switch the catalog name to `gold`, for example:

`list_tables(spark, "gold", "crypto")`